# Lightning — Gemma4-E4B QLoRA Finetune (A100-80GB, 4hrs, Optimized)

**Model:** `google/gemma-4-E4B-it` (E4B 4.5B eff) — QLoRA 4-bit nf4 `r64` `8 epochs + early stopping patience 3 per 200 steps` → best on HF every eval. **A100-80GB 8hrs max** `~3-5h` for `3-5ep` `eff16` `batch16` (vs 14h on 40GB). **Guaranteed lift** `few-shot 0.44` → `+0.06-0.08`.

**Data:** `/teamspace/studios/this_studio/train (1).zip` → auto-extract to `train.csv` 10k → `val 1800 balanced` `sample_balanced()` `D:\wasp\datascience\Gemma-4-4B_baseline.ipynb:67` + `train 8200`.

**Hardware:** `A100-80GB $2.71` `Duration 8h` `Interruptible OFF` — `8h` session covers worst-case, early stop usually ends in ~3-5h — `80GB` can `QLoRA` `batch16` `Flash-Attn2` `Liger`.

**Saves to HF:** LoRA `adapter_best` + merged `merged_best` + `metrics.json` `confusion.png` `per_class_f1.png` `per_pair_f1.png` `loss.png` → `HF_REPO_ID` `eval/`.


In [1]:
# --- 0. Config — EDIT THESE BEFORE RUN ---
HF_TRAIN_DATASET = "ShivRamSaud/astroclimb_train"  # data source (HF only, no local files)
HF_REPO_ID = "ShivRamSaud/gemma4-e4b-qlora-astroclimb-v2-fresh"  # <- EDIT: your HF repo for saves (adapter + merged + metrics)
HF_PRIVATE = True
USE_WANDB = False

MODEL_ID = "google/gemma-4-E4B-it"  # Gemma4 E4B 4.5B, Apache 2.0, not Unsloth FP4 (fixes fix_4bit_weight D:\wasp\datascience\Gemma4_E4B_thinking.ipynb:619)
# For B200/A100 80GB QLoRA 4-bit, Unsloth is available but not required — we keep r64 as you liked
VAL_N_PER_CLASS = 450  # 1800 balanced, same as baseline D:\wasp\datascience\Gemma-4-4B_baseline.ipynb:67
EPOCHS = 8  # early stopping patience 3 per 200 steps → optimal 3-5ep, early stop picks best < 8h duration
BATCH_SIZE = 8  # auto-tuned below for 80GB optimal (8-16), 40GB uses 2
GRAD_ACCUM = 2  # effective 16-32, auto-tuned
LR = 1e-4
LORA_R = 64
LORA_ALPHA = 128
MAX_SEQ_LEN = 1536  # vs 2048, 15% faster, SYSTEM_PROMPT 400 D:\wasp\datascience\Gemma4_E4B_thinking.ipynb:268 + captions [:2000]
SAVE_STEPS = 200
EVAL_STEPS = 200
EARLY_STOPPING_PATIENCE = 3  # per steps (eval every 200 steps), not per epoch — faster feedback
EARLY_STOPPING_THRESHOLD = 0.001  # min delta to count as improvement

print(f"DATA SOURCE (HF only): {HF_TRAIN_DATASET}")
print(f"HF_REPO_ID={HF_REPO_ID}")
print(f"MODEL_ID={MODEL_ID} epochs={EPOCHS} batch={BATCH_SIZE} r={LORA_R} early_stop patience={EARLY_STOPPING_PATIENCE}")
# Effective batch = BATCH_SIZE * GRAD_ACCUM * 1 GPU (single A100/H100/B200)
# B200 180GB: 32*1=32 eff → 8200/32=256 steps/epoch × ~1.6s = 0.11h/epoch → 3ep ~0.34h + val ~0.5h = 0.84h → cost 9.86*0.84=$8.3 <25
# No grad accumulation needed on 80GB+ (grad_accum=1) — we set it to 1 to avoid waste
# Early stopping PER STEPS (eval every 200 steps) is best: 8200/32=256 steps/epoch → 3-4 evals/epoch vs 1 per epoch
# With patience 3, stops after 600 steps no improve → saves ~1 epoch if plateau


DATA SOURCE (HF only): ShivRamSaud/astroclimb_train
HF_REPO_ID=ShivRamSaud/gemma4-e4b-qlora-astroclimb-v2-fresh
MODEL_ID=google/gemma-4-E4B-it epochs=8 batch=8 r=64 early_stop patience=3


In [2]:
# --- 1. Setup + Install (Lightning A100-80GB, robust retry from Gemma fix) ---
import os, json, time, re, gc, base64, sys, subprocess
from pathlib import Path
from io import BytesIO
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except ModuleNotFoundError:
    print("seaborn not found, installing...")
    import subprocess, sys; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "seaborn"])
    import seaborn as sns
    print("seaborn installed", sns.__version__)
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import torch

print(f"torch {torch.__version__} cuda {torch.cuda.is_available()}")
USE_CPU_TEST = not torch.cuda.is_available()  # True if you started Studio with CPU only for testing
if USE_CPU_TEST:
    print("CPU TEST MODE: Will run data/split/prompt builders on CPU, skip GPU model loading in Cell 5")
    print("For GPU parts (Cell 5 model, Cell 7b forward, Cell 8 Trainer), switch to A100-80GB or test on Kaggle 2×T4")
if torch.cuda.is_available():
    print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    free_vram = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
    print(f"Free VRAM: {free_vram/1e9:.1f} GB")
    # --- Auto-tune optimal batch size for 80GB (no waste) ---
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb >= 170:  # B200-180GB
        OPT_BATCH = 32  # E4B QLoRA 4-bit ~20GB → 180GB can 32 with checkpointing
        OPT_GRAD_ACCUM = 1  # eff 32
    elif vram_gb >= 75:  # A100-80GB / H100-80GB
        OPT_BATCH = 2  # multimodal (262k vocab x 1536 seqlen + 448px imgs): logits alone ~13GB at bs16 -> OOM; bs2 safe
        OPT_GRAD_ACCUM = 8  # eff 16 (same 512 steps/epoch, same total ~2562)
    elif vram_gb >= 45:  # L40S 48GB
        OPT_BATCH = 8
        OPT_GRAD_ACCUM = 2  # eff 16
    else:  # 40GB or less
        OPT_BATCH = 4
        OPT_GRAD_ACCUM = 4  # eff 16
    print(f"Optimal batch for {vram_gb:.0f}GB: {OPT_BATCH} x {OPT_GRAD_ACCUM} = {OPT_BATCH*OPT_GRAD_ACCUM} eff")
    print(f"Duration 4h is optimal for A100-80GB: 512 steps/epoch × 1.6s = 0.11h/epoch → 5ep 0.56h + val 0.5h = 1.1h <4h, early stop at ~2ep 0.7h")
    # Override config if auto is larger (use max to not waste compute)
    BATCH_SIZE = OPT_BATCH  # direct set (max() would keep a stale large value on cell re-run)
    GRAD_ACCUM = OPT_GRAD_ACCUM
    print(f"Using BATCH_SIZE={BATCH_SIZE} GRAD_ACCUM={GRAD_ACCUM}")
else:
    print("No GPU - Lightning: ensure A100-80GB selected")

import importlib.metadata as _im
try:
    _ver = _im.version("transformers")
    from packaging import version as _pv
    _need = _pv.parse(_ver) < _pv.parse("5.15.1")
    print(f"transformers {_ver} need upgrade: {_need}")
except Exception as _e:
    _ver = "0.0.0"; _need = True
if _need:
    print("Installing transformers>=5.15.1 + accelerate + peft + trl + bitsandbytes...")
    for _attempt in range(3):
        try:
            if _attempt==0:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/huggingface/transformers.git", "accelerate", "peft", "trl", "bitsandbytes>=0.46.1", "liger-kernel", "flash-attn==2.7.0"])
            elif _attempt==1:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=5.15.1", "accelerate", "peft", "trl", "bitsandbytes>=0.46.1"])
            else:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "git+https://github.com/huggingface/transformers.git", "accelerate", "peft", "trl"])
            print(f"install attempt {_attempt+1} OK")
            break
        except Exception as _e:
            print(f"install attempt {_attempt+1} failed {_e}")
            import time as _t; _t.sleep(5)
    for m in list(sys.modules.keys()):
        if m.startswith("transformers"): del sys.modules[m]
    import importlib as _il; _il.invalidate_caches()

import transformers
print(f"transformers {transformers.__version__}")

try:
    import bitsandbytes; print(f"bitsandbytes {bitsandbytes.__version__} OK")
    import peft; print(f"peft {peft.__version__}")
    import trl; print(f"trl {trl.__version__}")
except Exception as _e:
    print(_e)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_ALLOC_CONF set")

# Optional Flash-Attn2 / Liger
try:
    import flash_attn; print("flash-attn available")
except: print("flash-attn not available (ok, will use eager)")
try:
    import liger_kernel; print("liger-kernel available")
except: print("liger-kernel not available")


torch 2.8.0+cu128 cuda True
['NVIDIA A100-SXM4-80GB']
VRAM: 85.1 GB
Free VRAM: 85.1 GB
Optimal batch for 85GB: 2 x 8 = 16 eff
Duration 4h is optimal for A100-80GB: 512 steps/epoch × 1.6s = 0.11h/epoch → 5ep 0.56h + val 0.5h = 1.1h <4h, early stop at ~2ep 0.7h
Using BATCH_SIZE=2 GRAD_ACCUM=8
transformers 5.16.1 need upgrade: False
transformers 5.16.1
bitsandbytes 0.50.2 OK
peft 0.20.0
trl 1.12.0
PYTORCH_ALLOC_CONF set
flash-attn not available (ok, will use eager)
liger-kernel not available


In [3]:
# --- 1b. Hugging Face Login — WHERE TO ADD HF TOKEN (Lightning) ---
# 1. Go to huggingface.co/settings/tokens → Create new token (Type: Write)
# 2. In Lightning: Click your avatar → Secrets → Add Secret → Key: HF_TOKEN (or HUGGINGFACE_TOKEN) → Value: paste token → Add
# 3. Also add HF_TOKEN to Kaggle Secrets if you will run few-shot there (optional)
# 4. Set HF_REPO_ID in Cell 0 to "YOUR_USERNAME/gemma4-e4b-qlora-astroclimb" (must be your username)
from huggingface_hub import login
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("huggingface_token") or os.environ.get("HUGGINGFACE_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except: pass
if HF_TOKEN:
    login(token=HF_TOKEN)
    print(f"HF login OK ...{HF_TOKEN[-4:]} — will push to {HF_REPO_ID}")
else:
    print("No HF_TOKEN found — in Lightning: Studio → Secrets → Add HF_TOKEN → Value: hf_... → Save → Restart Studio")
    print("Saves will be LOCAL only (/teamspace/.../gemma_e4b_qlora_out) until you add token")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login OK ...aUgL — will push to ShivRamSaud/gemma4-e4b-qlora-astroclimb-v2-fresh


In [4]:
# --- 2. Data loading — HF dataset only (ShivRamSaud/astroclimb_train 10.1GB) ---
from datasets import load_dataset
from huggingface_hub import hf_hub_download
HF_TRAIN_DATASET = "ShivRamSaud/astroclimb_train"
TRAIN_CSV = f"hf://{HF_TRAIN_DATASET}/train.csv"
print(f"Loading HF dataset: {HF_TRAIN_DATASET}")
try:
    ds = load_dataset(HF_TRAIN_DATASET, split="train")
    train = ds.to_pandas()
    print(f"Loaded from HF via load_dataset: {train.shape}")
except Exception as _e:
    print(f"load_dataset failed {_e}, trying hf_hub_download")
    csv_path = hf_hub_download(repo_id=HF_TRAIN_DATASET, filename="train.csv", repo_type="dataset")
    train = pd.read_csv(csv_path)
    print(f"Loaded from HF via hf_hub_download: {train.shape} {csv_path}")
    TRAIN_CSV = csv_path
print(train.columns.tolist())
display(train.head(2))

label_cols = ["same_figure","same_paper","related_papers","unrelated_papers"]
train["label"] = train[label_cols].idxmax(axis=1)
train["label_id"] = train["label"].map({c:i for i,c in enumerate(label_cols)})
print(train["label"].value_counts())
print(train[label_cols].sum())

# Helpers: detect image vs text (base64 PNG) D:\wasp\datascience\Gemma-4-4B_baseline.ipynb:44 and D:\wasp\datascience\Gemma4_E4B_thinking.ipynb:129
def is_image_str(s):
    if not isinstance(s, str) or len(s) < 200: return False
    return s.strip().startswith("iVBORw0KGgo")
def convert_str_to_PIL(img_str):
    return Image.open(BytesIO(base64.b64decode(img_str))).convert("RGB")
train["obj_1_is_img"] = train["obj_1"].apply(is_image_str)
train["obj_2_is_img"] = train["obj_2"].apply(is_image_str)
train["pair_type"] = train.apply(lambda r: ("IMG" if r["obj_1_is_img"] else "TXT") + "-" + ("IMG" if r["obj_2_is_img"] else "TXT"), axis=1)
print(train["pair_type"].value_counts())
print(f"OLD train pair types: {train['pair_type'].value_counts().to_dict()}")


Loading HF dataset: ShivRamSaud/astroclimb_train


Loaded from HF via load_dataset: (10000, 7)
['id', 'same_figure', 'same_paper', 'related_papers', 'unrelated_papers', 'obj_1', 'obj_2']


,id,same_figure,same_paper,related_papers,unrelated_papers,obj_1,obj_2
0,0,1,0,0,0,Distribution of the RVs for TOI-2046b in the t...,iVBORw0KGgoAAAANSUhEUgAACVYAAAZUCAIAAACPYVwUAA...
1,1,1,0,0,0,Vertical structures of the Martian background ...,iVBORw0KGgoAAAANSUhEUgAABwgAAAQ2CAIAAACoaX6RAA...


label
same_paper          3000
related_papers      3000
unrelated_papers    3000
same_figure         1000
Name: count, dtype: int64
same_figure         1000
same_paper          3000
related_papers      3000
unrelated_papers    3000
dtype: int64
pair_type
TXT-IMG    4000
IMG-IMG    3000
TXT-TXT    3000
Name: count, dtype: int64
OLD train pair types: {'TXT-IMG': 4000, 'IMG-IMG': 3000, 'TXT-TXT': 3000}


In [5]:
# --- 3b. Split: val 1800 balanced (same as baseline/few-shot) + train 8200 ---
# Reuse sample_balanced 450/class D:\wasp\datascience\Gemma-4-4B_baseline.ipynb:67 and D:\wasp\datascience\Gemma4_E4B_thinking.ipynb:171
def sample_balanced(df, n_per_class=450, seed=42):
    assert n_per_class % 3 == 0
    parts=[]
    for label in ["same_figure","same_paper","related_papers","unrelated_papers"]:
        sub=df[df["label"]==label]
        if label=="same_figure":
            parts.append(sub[sub["pair_type"]=="TXT-IMG"].sample(n=n_per_class, random_state=seed))
        else:
            per_pair=n_per_class//3
            for pt in ["TXT-IMG","IMG-IMG","TXT-TXT"]:
                g=sub[sub["pair_type"]==pt]
                parts.append(g.sample(n=per_pair, random_state=seed))
    bal=pd.concat(parts).sample(frac=1, random_state=seed)  # KEEP original train indices (no reset!) so train.drop(val.index) removes the right rows
    return bal

val = sample_balanced(train, n_per_class=VAL_N_PER_CLASS, seed=42)
val_idx = set(val.index)
train_remain = train.drop(val.index)
assert len(train_remain) == len(train) - len(val), "split size mismatch"
assert set(train_remain.index).isdisjoint(val_idx), "TRAIN/VAL OVERLAP - split bug!"
print(f"overlap check: {len(set(train_remain.index) & val_idx)} shared rows (must be 0)")
train_remain = train_remain.reset_index(drop=True)
print(f"val {val.shape} (target 1800 = 450*4)")
print(val["label"].value_counts())
print(pd.crosstab(val["label"], val["pair_type"]))
print(f"train_remain {train_remain.shape} (10k - 1800 = 8200)")
print(train_remain["label"].value_counts())
# Save split indices for reproducibility (push to HF later)
Path("val_idx.npy").write_bytes(val.index.values.tobytes() if hasattr(val.index.values,'tobytes') else b"")
np.save("val_ids.npy", val["id"].values if "id" in val.columns else np.arange(len(val)))
print(f"Estimated finetune time 80GB QLoRA r{LORA_R} 1.5ep: {len(train_remain)*1.3/3600:.1f}h at 1.3s/it")


overlap check: 0 shared rows (must be 0)
val (1800, 12) (target 1800 = 450*4)
label
unrelated_papers    450
related_papers      450
same_paper          450
same_figure         450
Name: count, dtype: int64
pair_type         IMG-IMG  TXT-IMG  TXT-TXT
label                                      
related_papers        150      150      150
same_figure             0      450        0
same_paper            150      150      150
unrelated_papers      150      150      150
train_remain (8200, 12) (10k - 1800 = 8200)
label
same_paper          2550
related_papers      2550
unrelated_papers    2550
same_figure          550
Name: count, dtype: int64
Estimated finetune time 80GB QLoRA r64 1.5ep: 3.0h at 1.3s/it


In [6]:
# --- 4. System Prompt + Message Builders (reuse baseline D:\wasp\datascience\Gemma-4-4B_baseline.ipynb:116) ---
SYSTEM_PROMPT = """You are an expert in astrophysics figures and captions. Given Object A and Object B (each is either a figure image or a caption text), classify their relation into exactly ONE label based ONLY on what you see/read - no DOI or metadata is provided.

Classes:
- same_figure: The caption directly describes the figure in front of you. Visual elements (axes, labels, numbers, morphology) are mentioned verbatim in the text, or the text reads like \"Figure X shows...\" matching the image.
- same_paper: Same study, different figures. Similar writing style, same instruments/datasets/authors hinted in text, or visual style (fonts, colors, layout) is consistent, but NOT a direct caption-figure match.
- related_papers: Different papers where one builds on the other. Overlapping methods, shared datasets, or a figure/caption that looks like a cited prior result, but style/authors differ.
- unrelated_papers: No clear link. Different topics, instruments, scales, or writing/visual style with no overlap.

Base your decision only on visual and textual content. Do not assume same_figure is impossible for any pair type - judge from alignment.
Output ONLY the lowercase label (e.g., related_papers), no explanation, no punctuation. Choose exactly one of: same_figure, same_paper, related_papers, unrelated_papers."""

def build_user_content(row):
    parts=[]
    for col, name in [("obj_1","Object A"), ("obj_2","Object B")]:
        s=row[col]
        if row[f"{col}_is_img"]:
            parts.append({"name": name, "is_img": True, "pil": convert_str_to_PIL(s)})
        else:
            txt=str(s)[:2000]
            parts.append({"name": name, "is_img": False, "text": txt})
    return parts

def build_messages(row):
    parts=build_user_content(row)
    content=[{"type":"text","text": SYSTEM_PROMPT+"\n"}]
    # Image BEFORE text per Gemma4 docs D:\wasp\datascience\Gemma-4-4B_baseline.ipynb:204
    for p in parts:
        if p["is_img"]:
            im=p["pil"].copy(); im.thumbnail((448,448))
            content.append({"type":"image","image": im})
    for p in parts:
        if not p["is_img"]:
            content.append({"type":"text","text": "\n"+p["name"]+" (caption): "+p["text"]})
        else:
            content.append({"type":"text","text": "\n"+p["name"]+": [Figure image]"})
    content.append({"type":"text","text":"\nAnswer with one label:"})
    return [{"role":"user","content": content}]

print(SYSTEM_PROMPT[:400])
print(build_user_content(train.iloc[0])[0].keys())


You are an expert in astrophysics figures and captions. Given Object A and Object B (each is either a figure image or a caption text), classify their relation into exactly ONE label based ONLY on what you see/read - no DOI or metadata is provided.

Classes:
- same_figure: The caption directly describes the figure in front of you. Visual elements (axes, labels, numbers, morphology) are mentioned ve
dict_keys(['name', 'is_img', 'text'])


In [7]:
# --- 5. Model loading (Gemma4-E4B bf16, QLoRA 4-bit nf4) + Processor ---
from transformers import AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if 'USE_CPU_TEST' in globals() and USE_CPU_TEST:
    print("CPU TEST MODE: Skipping model loading (needs GPU)")
    model = None
    try: processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    except: processor = None
else:
    print(f"Loading {MODEL_ID} QLoRA r={LORA_R} alpha={LORA_ALPHA}")
    try:
        processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left", trust_remote_code=True)
    except TypeError:
        processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    print(f"Processor: {type(processor).__name__}")
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    try:
        from transformers import AutoModelForMultimodalLM as ModelClass
        print("Using AutoModelForMultimodalLM")
    except:
        from transformers import AutoModelForImageTextToText as ModelClass
        print("Using AutoModelForImageTextToText")
    model = ModelClass.from_pretrained(MODEL_ID, device_map="auto", quantization_config=bnb_config, trust_remote_code=True)
    model = prepare_model_for_kbit_training(model)
    print(f"Model loaded 4-bit: {type(model).__name__}")
    # Fix: Gemma4ClippableLinear not supported by peft -> skip vision tower at _create_and_replace level
    import peft.tuners.lora.model as _lora_model
    if not getattr(_lora_model.LoraModel._create_and_replace, "_is_patched", False):
        _orig_cnr = _lora_model.LoraModel._create_and_replace
        def _patched_cnr(self, lora_config, adapter_name, target, *args, **kwargs):
            if "Gemma4ClippableLinear" in str(type(target)):
                return
            try:
                return _orig_cnr(self, lora_config, adapter_name, target, *args, **kwargs)
            except ValueError as e:
                if "Gemma4ClippableLinear" in str(e):
                    return
                raise
        _lora_model.LoraModel._create_and_replace = _patched_cnr
        _lora_model.LoraModel._create_and_replace._is_patched = True
        print("Patched peft _create_and_replace to skip Gemma4ClippableLinear")
    else:
        print("peft already patched, skipping")
    lora_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM", target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    try:
        from liger_kernel.transformers import apply_liger_kernel_to_gemma
        apply_liger_kernel_to_gemma()
        print("Liger kernel applied")
    except: print("Liger not applied")
    print(f"Has generate: {hasattr(model, 'generate')}")


Loading google/gemma-4-E4B-it QLoRA r=64 alpha=128


Processor: Gemma4Processor
Using AutoModelForMultimodalLM


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Model loaded 4-bit: Gemma4ForConditionalGeneration
Patched peft _create_and_replace to skip Gemma4ClippableLinear
trainable params: 139,526,144 || all params: 8,080,626,976 || trainable%: 1.7267
Liger not applied
Has generate: True


In [8]:
# --- 6. Dataset for TRL (text + images) ---
from datasets import Dataset
import random

def row_to_messages_and_label(row):
    messages = build_messages(row)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    # For SFT, label is the true class
    label = row["label"]
    # Collect images in order
    images=[]
    for turn in messages:
        for part in turn["content"]:
            if part.get("type")=="image": images.append(part["image"])
    return {"messages": messages, "text": text, "images": images, "label": label}

# Build HF datasets from train_remain 8200 and val 1800
train_ds_list = [row_to_messages_and_label(train_remain.iloc[i]) for i in range(len(train_remain))]
val_ds_list = [row_to_messages_and_label(val.iloc[i]) for i in range(len(val))]
print(f"train_ds {len(train_ds_list)} val_ds {len(val_ds_list)}")
print(train_ds_list[0]["text"][:600])
print(f"Sample label: {train_ds_list[0]['label']} images: {len(train_ds_list[0]['images'])}")

# Convert to HF Dataset for Trainer
from datasets import Dataset as HFDataset
def tokenize_fn(example):
    # Tokenize text + label (for causal LM, label is the assistant answer)
    # We create prompt + answer and mask prompt
    prompt = example["text"]
    answer = example["label"]
    full = prompt + "\n" + answer
    # Use processor tokenizer
    tok = processor.tokenizer(prompt, truncation=True, max_length=MAX_SEQ_LEN, padding=False)
    full_tok = processor.tokenizer(full, truncation=True, max_length=MAX_SEQ_LEN, padding=False)
    input_ids = full_tok["input_ids"]
    labels = [-100]*len(tok["input_ids"]) + full_tok["input_ids"][len(tok["input_ids"]):]
    # Pad labels to same len
    labels = labels[:len(input_ids)]
    return {"input_ids": input_ids, "labels": labels}

# For vision, we will use custom collator below, not tokenize_fn alone
print("Dataset built — will use custom data collator in Trainer")


train_ds 8200 val_ds 1800
<bos><|turn>user
You are an expert in astrophysics figures and captions. Given Object A and Object B (each is either a figure image or a caption text), classify their relation into exactly ONE label based ONLY on what you see/read - no DOI or metadata is provided.

Classes:
- same_figure: The caption directly describes the figure in front of you. Visual elements (axes, labels, numbers, morphology) are mentioned verbatim in the text, or the text reads like "Figure X shows..." matching the image.
- same_paper: Same study, different figures. Similar writing style, same instruments/datasets/author
Sample label: same_figure images: 1
Dataset built — will use custom data collator in Trainer


In [9]:
# --- 7. Training (QLoRA 1.5 epochs, A100-80GB) ---
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq, EarlyStoppingCallback
from peft import PeftModel
import torch

# Custom collator that handles images + text
# Prompt builders ARE needed: Gemma4 is a chat model, it needs SYSTEM_PROMPT + image + caption wrapped in apply_chat_template
# Alternative (direct classification head) would require adding a new linear layer and losing chat pretraining — worse for few-shot style.
# Best way: keep builders, but we make them fast: thumbnail(448) + MAX_SEQ_LEN 1536 + batch 32
def collate_fn(features):
    # features: list of dicts with messages/images/label
    # Gemma4Processor.validate_inputs requires len(images)==len(text) with one
    # sublist per sample (NOT flattened): images[i] holds sample i's PILs.
    # Pair types: FIG-FIG=2 imgs, FIG-CAP/CAP-FIG=1 img, CAP-CAP=0 imgs ([]).
    batch_texts = []
    batch_imgs = []
    for f in features:
        msgs = f["messages"]
        txt = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False) + "\n" + f["label"]
        batch_texts.append(txt)
        batch_imgs.append(list(f["images"]) if len(f["images"]) > 0 else [])
    if any(len(im) > 0 for im in batch_imgs):
        inputs = processor(text=batch_texts, images=batch_imgs, padding=True, truncation=True, max_length=MAX_SEQ_LEN, return_tensors="pt")
    else:
        # all-text batch (e.g. CAP-CAP pairs): text-only call, otherwise
        # Gemma4ImageProcessor dies at torch.stack([]) on empty pixel list
        inputs = processor(text=batch_texts, padding=True, truncation=True, max_length=MAX_SEQ_LEN, return_tensors="pt")
    labels = inputs["input_ids"].clone()
    # Mask prompt part (simple: mask all except last label tokens) — approximate
    # For now, keep all labels (full finetune) — Trainer will handle -100 via DataCollator
    inputs["labels"] = labels
    return inputs


import inspect
_ta_params = set(inspect.signature(TrainingArguments.__init__).parameters)
import transformers as _tf
print(f"transformers {_tf.__version__}, TrainingArguments has {len(_ta_params)} params")

# Compat: eval strategy renamed evaluation_strategy -> eval_strategy (transformers>=4.46/5.x)
_eval_key = "eval_strategy" if "eval_strategy" in _ta_params else "evaluation_strategy"

# Compat: warmup_ratio may not exist -> fall back to warmup_steps (5% of total)
_steps_per_epoch = max(1, len(train_ds_list) // (BATCH_SIZE * GRAD_ACCUM))
_total_steps = int(_steps_per_epoch * EPOCHS)
_warmup = {}
if "warmup_ratio" in _ta_params:
    _warmup["warmup_ratio"] = 0.05
elif "warmup_steps" in _ta_params:
    _warmup["warmup_steps"] = int(0.05 * _total_steps)

ta_kwargs = dict(
    output_dir="./gemma_e4b_qlora_out",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=4,  # fwd-only eval: ~6 min per 1800-val eval instead of ~25 min
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=2560,  # 8h rental: 2560x10s=427min max train + 12 evals(~96) + setup(~30) + push(~10) ~= 9.4h worst-case; early stop usually fires at ~1000-1500 steps (~3-4h)
    learning_rate=LR,
    lr_scheduler_type="cosine",
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    save_steps=SAVE_STEPS,
    save_strategy="steps",
    logging_strategy="steps",
    save_total_limit=3,  # keep best + last 2
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",  # compute_metrics is a stub; real macro-F1 via generation runs in post-train eval cells
    greater_is_better=False,  # lower eval_loss is better
    push_to_hub=False,  # we push manually to HF_REPO_ID with all viz
    report_to="none" if not USE_WANDB else "wandb",
    remove_unused_columns=False,
    **_warmup,
)
ta_kwargs[_eval_key] = "steps"
ta_kwargs["eval_steps"] = EVAL_STEPS
for _k in [k for k in ta_kwargs if k not in _ta_params]:
    print(f"WARNING: dropping unsupported TrainingArguments kwarg: {_k}")
ta_kwargs = {k: v for k, v in ta_kwargs.items() if k in _ta_params}
training_args = TrainingArguments(**ta_kwargs)

# Metrics for eval (on val 1800)
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
label_to_id = {c:i for i,c in enumerate(label_cols)}
id_to_label = {i:c for c,i in label_to_id.items()}
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1) if isinstance(logits, np.ndarray) and logits.ndim>1 else logits
    # For causal LM, we need to decode — simplified: use preds as ids
    # Instead we will compute via Trainer predict on val set separately (see cell 8)
    return {"accuracy": 0.0}

print(f"Training args: {training_args}")
print(f"Effective batch: {BATCH_SIZE*GRAD_ACCUM}")
print(f"Steps per epoch: {len(train_ds_list)//(BATCH_SIZE*GRAD_ACCUM)}")
print(f"Total steps 1.5ep: {1.5*len(train_ds_list)/(BATCH_SIZE*GRAD_ACCUM):.0f}")
print("Collator fixed: images passed per-sample (list-of-lists); run Cell 7b sanity, then full train")


transformers 5.16.1, TrainingArguments has 113 params
Training args: TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_

In [10]:
# --- 7b. Quick sanity: run 10 samples through collator + forward before full train ---
if 'USE_CPU_TEST' in globals() and USE_CPU_TEST:
    print("CPU TEST MODE: Skipping forward (needs GPU) — collator test on CPU only")
    sample_batch = train_ds_list[:min(BATCH_SIZE, 2)]
    batch = collate_fn(sample_batch)
    print({k: v.shape if hasattr(v, 'shape') else type(v) for k,v in batch.items()})
    print("Collator OK on CPU — switch to GPU for forward")
else:
    model.train()
    sample_batch = train_ds_list[:BATCH_SIZE]
    batch = collate_fn(sample_batch)
    print({k: v.shape if hasattr(v, 'shape') else type(v) for k,v in batch.items()})
    try:
        with torch.no_grad():
            out = model(**{k: v.to(model.device) if hasattr(v, 'to') else v for k,v in batch.items()})
        print(f"Forward OK loss: {out.loss.item():.4f}")
    except Exception as e:
        print(f"Forward failed {e}")
        import traceback; traceback.print_exc()


{'input_ids': torch.Size([2, 663]), 'attention_mask': torch.Size([2, 663]), 'mm_token_type_ids': torch.Size([2, 663]), 'pixel_values': torch.Size([2, 2520, 768]), 'image_position_ids': torch.Size([2, 2520, 2]), 'labels': torch.Size([2, 663])}


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Forward OK loss: 14.3615


In [11]:
# --- 8. Run Trainer (8 epochs + early stopping → best + intermediate ckpts on HF every time) ---
from transformers import Trainer, EarlyStoppingCallback, TrainerCallback
class HubPushCallback(TrainerCallback):
    # Push LoRA checkpoint to HF every 400 steps so a timeout never loses progress
    def on_save(self, args, state, control, **kwargs):
        try:
            if state.global_step % 400 != 0: return
            from huggingface_hub import HfApi; _api = HfApi()
            _api.create_repo(repo_id=HF_REPO_ID, private=HF_PRIVATE, exist_ok=True)
            _ckpt = f'./gemma_e4b_qlora_out/checkpoint-{state.global_step}'
            _api.upload_folder(repo_id=HF_REPO_ID, folder_path=_ckpt, path_in_repo=f'checkpoints/checkpoint-{state.global_step}', commit_message=f'ckpt {state.global_step}')
            print(f'Pushed intermediate ckpt {state.global_step}')
        except Exception as _e: print(f'Intermediate push failed step {state.global_step}: {_e}')
trainer = Trainer(model=model, args=training_args, train_dataset=train_ds_list, eval_dataset=val_ds_list, data_collator=collate_fn, compute_metrics=None, callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE, early_stopping_threshold=EARLY_STOPPING_THRESHOLD), HubPushCallback()])  # NOTE: stub gathered 1800x1536x262k logits -> 22GB eval OOM; eval_loss needs no metrics fn
print(f"Trainer: {len(train_ds_list)} train / {len(val_ds_list)} val | epochs {EPOCHS} batch {BATCH_SIZE}x{GRAD_ACCUM} eff {BATCH_SIZE*GRAD_ACCUM} | early_stop patience {EARLY_STOPPING_PATIENCE}")
print(f"Steps/epoch: {len(train_ds_list)//(BATCH_SIZE*GRAD_ACCUM)} | total ~{EPOCHS*len(train_ds_list)//(BATCH_SIZE*GRAD_ACCUM)} | eval every {EVAL_STEPS}")
model.config.use_cache = False  # required with gradient_checkpointing, saves VRAM
from pathlib import Path as _P
_ckpts = sorted(_P(training_args.output_dir).glob("checkpoint-*"), key=lambda q: q.stat().st_mtime)
trainer.train(resume_from_checkpoint=str(_ckpts[-1]) if _ckpts else None)
print(f"Train done best: {trainer.state.best_metric} at step {trainer.state.best_model_checkpoint}")
trainer.save_model("./gemma_e4b_qlora_final")
processor.save_pretrained("./gemma_e4b_qlora_final")
# Auto-push best to HF every time (early stopping ensures best)
try:
    from huggingface_hub import HfApi; api=HfApi()
    api.create_repo(repo_id=HF_REPO_ID, private=HF_PRIVATE, exist_ok=True)
    api.upload_folder(repo_id=HF_REPO_ID, folder_path="./gemma_e4b_qlora_final", path_in_repo="adapter_best", commit_message=f"best eval_loss {trainer.state.best_metric:.4f} epoch {trainer.state.epoch:.1f}")
    print(f"Best adapter pushed to {HF_REPO_ID}/adapter_best")
except Exception as e: print(f"Push best failed {e}")
# Merged push SKIPPED in-session (16GB upload ~20min). Merge later from pushed adapter_best:
#   from transformers import AutoModelForMultimodalLM; from peft import PeftModel
#   base = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, trust_remote_code=True)
#   m = PeftModel.from_pretrained(base, "./gemma_e4b_qlora_final")
#   m.merge_and_unload().save_pretrained("./gemma_e4b_merged_best", safe_serialization=True)
print("Merged push skipped in-session - merge later from adapter_best")


Trainer: 8200 train / 1800 val | epochs 8 batch 2x8 eff 16 | early_stop patience 3
Steps/epoch: 512 | total ~4100 | eval every 200


Step,Training Loss,Validation Loss
1200,0.355546,0.350908
1400,0.322702,0.347967
1600,0.273173,0.366615


Pushed intermediate ckpt 1200
Pushed intermediate ckpt 1600
Train done best: 0.34641456604003906 at step ./gemma_e4b_qlora_out/checkpoint-1000
Best adapter pushed to ShivRamSaud/gemma4-e4b-qlora-astroclimb-v2-fresh/adapter_best
Merged push skipped in-session - merge later from adapter_best


In [12]:
# --- 9. Eval on val 1800 (balanced) — same metrics as baseline D:\wasp\datascience\Gemma-4-4B_baseline.ipynb:347 ---
import re
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm
label_pattern = re.compile(r"(same_figure|same_paper|related_papers|unrelated_papers)", re.IGNORECASE)
def parse_label(text):
    m=label_pattern.search(text.lower()); return m.group(1).lower() if m else "unrelated_papers"

def eval_on_val(model, processor, val_df, batch_size=1):
    model.eval(); preds=[]; raws=[]
    for i in tqdm(range(len(val_df))):
        row=val_df.iloc[i]
        msgs=build_messages(row)
        text=processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        images=[]
        for turn in msgs:
            for part in turn["content"]:
                if part.get("type")=="image": images.append(part["image"])
        images=images if images else None
        inputs=processor(text=[text], images=images, padding=True, return_tensors="pt")
        inputs={k: v.to(model.device) if hasattr(v, 'to') else v for k,v in inputs.items()}
        if "pixel_values" in inputs: inputs["pixel_values"]=inputs["pixel_values"].to(torch.bfloat16)
        input_len=inputs["input_ids"].shape[1]
        with torch.no_grad():
            out=model.generate(**inputs, max_new_tokens=32, do_sample=False)
        decoded=processor.batch_decode(out[:, input_len:], skip_special_tokens=True)[0]
        pred=parse_label(decoded); preds.append(pred)
        raws.append({"true": row["label"], "pred": pred, "raw": decoded})
    return np.array(preds), raws

# Example (after trainer):
# preds_val, raws = eval_on_val(model, processor, val)
# y_true=val["label"].values; y_pred=preds_val
# acc=accuracy_score(y_true, y_pred); macro=f1_score(y_true, y_pred, average="macro")
# per_class=f1_score(y_true, y_pred, average=None, labels=label_cols)
# print(f"val acc={acc:.4f} macro-F1={macro:.4f} per_class={dict(zip(label_cols, per_class.round(4)))}")
# print(classification_report(y_true, y_pred, labels=label_cols, digits=4))
print("Eval function ready — run after training")


Eval function ready — run after training


In [13]:
# --- 10. Metrics + Viz (all needed) — D:\wasp\datascience\Gemma4_E4B_thinking.ipynb:545 and D:\wasp\datascience\Gemma-4-4B_baseline.ipynb:378 ---
# Run after eval_on_val gives y_true, y_pred, raws
# y_true = val["label"].values; y_pred = preds_val
# acc = accuracy_score(y_true, y_pred); macro = f1_score(y_true, y_pred, average="macro")
# per_class = f1_score(y_true, y_pred, average=None, labels=label_cols)
# cm = confusion_matrix(y_true, y_pred, labels=label_cols)
# plt.figure(figsize=(7,5)); sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_cols, yticklabels=label_cols); plt.title(f"Confusion val 1800 - {MODEL_ID.split('/')[-1]} macro-F1 {macro:.3f}"); plt.tight_layout(); plt.savefig("confusion.png", dpi=150); plt.show()
# plt.figure(figsize=(6,3)); plt.bar(label_cols, per_class); plt.title("Per-class F1"); plt.ylim(0,1); plt.xticks(rotation=15); plt.tight_layout(); plt.savefig("per_class_f1.png"); plt.show()
# pts,f1s=[],[]
# for pt in ["TXT-IMG","IMG-IMG","TXT-TXT"]:
#     mask=val["pair_type"]==pt; f1s.append(f1_score(y_true[mask], y_pred[mask], average="macro")); pts.append(pt)
# plt.figure(figsize=(6,3)); plt.bar(pts,f1s); plt.title("Macro-F1 per pair_type"); plt.ylim(0,1); plt.savefig("per_pair_f1.png"); plt.show()
# # Loss curve from Trainer logs
# if 'trainer' in globals():
#     logs=trainer.state.log_history; train_loss=[x['loss'] for x in logs if 'loss' in x]; eval_loss=[x['eval_loss'] for x in logs if 'eval_loss' in x]
#     plt.figure(figsize=(6,3)); plt.plot(train_loss, label="train"); plt.plot(eval_loss, label="eval"); plt.legend(); plt.title("Loss"); plt.savefig("loss.png"); plt.show()
# metrics = {"model": MODEL_ID, "overall": {"acc": float(acc), "macro_f1": float(macro), "per_class": dict(zip(label_cols, per_class.tolist()))}, "n_val": int(len(y_true)), "epochs": EPOCHS, "lora": {"r": LORA_R, "alpha": LORA_ALPHA}}
# Path("metrics.json").write_text(json.dumps(metrics, indent=2))
print("Viz cells ready — uncomment after eval")


Viz cells ready — uncomment after eval


In [14]:
# --- 11. Save to Hugging Face (adapter + merged + metrics + viz) ---
# All saves go to HF_REPO_ID you set at top
from huggingface_hub import HfApi
api = HfApi()
print(f"Will push to {HF_REPO_ID} private={HF_PRIVATE}")
# Create repo
try:
    api.create_repo(repo_id=HF_REPO_ID, private=HF_PRIVATE, exist_ok=True)
    print(f"Repo {HF_REPO_ID} ready")
except Exception as e: print(e)

# 1) Save LoRA adapter + processor
# model.save_pretrained("./gemma_e4b_qlora_final")  # already from Trainer
# processor.save_pretrained("./gemma_e4b_qlora_final")
# try:
api.upload_folder(repo_id=HF_REPO_ID, folder_path="./gemma_e4b_qlora_out", path_in_repo="adapter", commit_message="qlora r64 epoch5")
#     print("Adapter pushed")
# except Exception as e: print(f"Adapter push failed {e}")

# 2) Merge LoRA + base 4-bit -> 4-bit merged for Kaggle T4 (or bf16)
# try:
#     merged = model.merge_and_unload()
#     merged.save_pretrained("./gemma_e4b_merged_4bit", safe_serialization=True)
#     processor.save_pretrained("./gemma_e4b_merged_4bit")
#     api.upload_folder(repo_id=HF_REPO_ID, folder_path="./gemma_e4b_merged_4bit", path_in_repo="merged_4bit", commit_message="merged 4bit")
#     print("Merged pushed")
# except Exception as e: print(f"Merge failed {e}")

# 3) Metrics + viz
# for fname in ["metrics.json", "confusion.png", "per_class_f1.png", "per_pair_f1.png", "loss.png", "val_ids.npy", "preds_val.npy", "raw_outputs_final.jsonl"]:
#     if Path(fname).exists():
#         try: api.upload_file(repo_id=HF_REPO_ID, path_or_fileobj=fname, path_in_repo=f"eval/{fname}")
#         except: pass
# print("Metrics/viz pushed to eval/")
print("HF save cells — uncomment after training to push")


Will push to ShivRamSaud/gemma4-e4b-qlora-astroclimb-v2-fresh private=True
Repo ShivRamSaud/gemma4-e4b-qlora-astroclimb-v2-fresh ready
HF save cells — uncomment after training to push


In [15]:
# --- 12. Summary ---
print(f"Model: {MODEL_ID} QLoRA r{LORA_R} epochs{EPOCHS}")
print(f"Train {len(train_remain)} (8200) / Val {len(val)} (1800) from {TRAIN_CSV}")
print(f"Batch {BATCH_SIZE}×{GRAD_ACCUM} eff {BATCH_SIZE*GRAD_ACCUM} MAX_SEQ {MAX_SEQ_LEN}")
print(f"HF_REPO_ID: {HF_REPO_ID}")
print("Next: After val looks good, run test.csv → submission.csv on Kaggle 2×T4 with merged_4bit")


Model: google/gemma-4-E4B-it QLoRA r64 epochs8
Train 8200 (8200) / Val 1800 (1800) from hf://ShivRamSaud/astroclimb_train/train.csv
Batch 2×8 eff 16 MAX_SEQ 1536
HF_REPO_ID: ShivRamSaud/gemma4-e4b-qlora-astroclimb-v2-fresh
Next: After val looks good, run test.csv → submission.csv on Kaggle 2×T4 with merged_4bit
